# Start Here — Live Project Demo
## Agentic RAG for research reports about AI in healthcare
Adarsh Konderu

This is my **live generation entry point**, not a saved-results replay. I enter one question and run the existing Condition C workflow: plan → retrieve → review evidence → write → check and conditionally revise → audit claims.

The agents are different roles using the **same Qwen2.5-3B-Instruct model**, not five separate models. A separate DeBERTa NLI model checks claims. These checks are fallible and do not replace human evaluation or make the report suitable for clinical advice.

The starting data is the frozen collection of **46 PMC articles and 1,415 cleaned chunks**. The earlier collection/cleaning code is supplied in `stage_code`; this demo does not collect the articles again.

### How I run this in a meeting

1. In Colab, select a **T4 GPU** runtime, then run the cells in order.
2. Edit the question in the next cell. Keep `RUN_BASELINES = False` for the shorter Condition C demonstration.
3. When prompted, I upload the Stage 3.2 retrieval evidence ZIP from my approved project files.
4. Show the fresh report, selected passages, workflow trace and claim audit near the bottom.
5. Run the final download cell to preserve the new demo output. Use the separate Results and Graphs notebook for the frozen 45-report experiment.

Initial model downloads can take several minutes. This notebook needs internet and an available GPU; no paid API key is used. Free GPU availability is not guaranteed ([Colab FAQ](https://research.google.com/colaboratory/faq.html)). If unavailable, show saved results and explicitly call them saved results; do not claim a live run succeeded.

**Validation:** notebook syntax, input hashes and local non-GPU checks are tested. This newly assembled notebook still needs a full Colab GPU rehearsal before the meeting. It reuses the frozen generation code, but this is not a promise of error-free execution on a future runtime.


In [ ]:
# This is the only setting I normally change during a demonstration.
RESEARCH_QUESTION = "What hallucination and factual accuracy risks occur when healthcare reports use large language models?"
RUN_BASELINES = False  # True also runs A (direct) and B (agents without retrieval).
assert isinstance(RESEARCH_QUESTION, str) and RESEARCH_QUESTION.strip()
assert len(RESEARCH_QUESTION) <= 2000, "Please use a concise research question."


## 1. Install the project libraries

These version pins are inherited from the experimental notebooks. Run this before the imports. If Colab says a runtime restart is required, restart, rerun the question cell, and continue below. Do not upgrade packages mid-demo.


In [ ]:
# Qwen2.5 requires Transformers 4.37 or newer. These tested versions are
# pinned so that Colab does not silently change the software environment.
!pip -q install "transformers==4.49.0" "accelerate==1.3.0" "sentence-transformers==3.4.1" bitsandbytes

## 2. Imports and fixed settings

`import re` supplies text-pattern matching, for example finding citation labels. The other imports handle models, arrays, files and the recorded trace. A missing GPU stops here rather than silently pretending to run the model.

In [ ]:
import csv
import gc
import importlib.metadata
import io
import json
import math
import random
import re
import shutil
import sys
import zipfile
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
import transformers
from google.colab import files
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Separate input and output folders avoid accidentally changing the Stage 3 files.
INPUT_DIR = Path("/content/project_demo_input")
OUTPUT_DIR = Path("/content/project_demo_runs")
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# These retrieval values are carried forward from the Stage 3.1 comparison.
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_MODEL_REVISION = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"
GENERATOR_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
GENERATOR_MODEL_REVISION = "a1d308dfcc03e09da285d49d912439a655a571e8"
TOP_K_CANDIDATES = 5
MAXIMUM_PER_ARTICLE = 1
RRF_CONSTANT = 60
DENSE_WEIGHT = 0.65
BM25_WEIGHT = 0.35
BM25_K1 = 1.5
BM25_B = 0.75

# Fixed generation values reduce variation between repeated runs.
RANDOM_SEED = 42
PLANNER_MAX_TOKENS = 320
REVIEWER_MAX_TOKENS = 280
WRITER_MAX_TOKENS = 400
REPORT_REVIEW_MAX_TOKENS = 300
MAX_REPORT_SECTIONS = 4

# This development question is specifically about LLMs, not every type of medical AI.
DEVELOPMENT_EVIDENCE_THEMES = {
    "generative_ai_and_llms",
    "ethics_safety_and_bias",
}

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# Stop with a clear instruction if an older imported version is still in memory.
if transformers.__version__ != "4.49.0":
    raise RuntimeError(
        "Transformers 4.49.0 was installed but a different version is still loaded. "
        "Select Runtime > Disconnect and delete runtime, then run the notebook again."
    )

if not torch.cuda.is_available():
    raise RuntimeError(
        "A GPU is required. In Colab choose Runtime > Change runtime type > T4 GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("Transformers:", transformers.__version__)
print("Generator:", GENERATOR_MODEL_NAME)
print("Generator revision:", GENERATOR_MODEL_REVISION)
print("Retrieval weights:", DENSE_WEIGHT, "dense +", BM25_WEIGHT, "BM25")

## 3. Load the frozen article data

The research data is not stored in this public repository. This cell asks me to upload the approved Stage 3.2 evidence ZIP and verifies the internal SHA-256 checks before using it.

In [ ]:
import hashlib

uploaded_files = files.upload()

expected_name = "Adarsh_Konderu_Stage_3_2_Clean_Retrieval_Evidence_v2.zip"
if expected_name in uploaded_files:
    uploaded_name = expected_name
elif len(uploaded_files) == 1:
    uploaded_name = next(iter(uploaded_files))
else:
    raise ValueError("Please upload only the Stage 3.2 v2 evidence ZIP.")

archive_bytes = uploaded_files[uploaded_name]
source_archive_sha256 = hashlib.sha256(archive_bytes).hexdigest()

# Check every path before extraction to avoid unsafe ZIP contents.
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    for member in archive.namelist():
        member_path = Path(member)
        if member_path.is_absolute() or ".." in member_path.parts:
            raise ValueError(f"Unsafe archive path: {member}")
    archive.extractall(INPUT_DIR)

required_files = [
    "pmc_chunks_cleaned.jsonl",
    "pmc_chunk_embeddings_cleaned.npy",
    "stage_3_2_run_metadata.json",
    "checksums_sha256.json",
]
for required_file in required_files:
    assert (INPUT_DIR / required_file).exists(), f"Missing {required_file}"

# Verify the stored digest for every evidence file before using it.
stored_checksums = json.loads(
    (INPUT_DIR / "checksums_sha256.json").read_text(encoding="utf-8")
)
for file_name, expected_digest in stored_checksums.items():
    file_path = INPUT_DIR / file_name
    assert file_path.exists(), f"Checksum file is missing: {file_name}"
    actual_digest = hashlib.sha256(file_path.read_bytes()).hexdigest()
    assert actual_digest == expected_digest, f"Checksum mismatch: {file_name}"

print("Stage 3.2 v2 evidence and checksums verified")
print("Uploaded ZIP SHA-256:", source_archive_sha256)

In [ ]:
# Load each JSON line as one cleaned passage record.
chunks = [
    json.loads(line)
    for line in (INPUT_DIR / "pmc_chunks_cleaned.jsonl")
    .read_text(encoding="utf-8")
    .splitlines()
    if line.strip()
]
embeddings = np.load(INPUT_DIR / "pmc_chunk_embeddings_cleaned.npy")
stage_3_metadata = json.loads(
    (INPUT_DIR / "stage_3_2_run_metadata.json").read_text(encoding="utf-8")
)

# These checks stop the workflow if an older or incompatible package is used.
assert len(chunks) == 1415
assert embeddings.shape == (1415, 384)
assert len({chunk["chunk_id"] for chunk in chunks}) == 1415
assert len({chunk["pmcid"] for chunk in chunks}) == 46
assert np.isfinite(embeddings).all()
assert np.allclose(np.linalg.norm(embeddings, axis=1), 1.0, atol=1e-5)
assert stage_3_metadata["article_count"] == 46
assert stage_3_metadata["chunk_count"] == 1415
assert stage_3_metadata["cleaned_article_count"] == 2
assert stage_3_metadata["embedding_model"] == EMBEDDING_MODEL_NAME
assert stage_3_metadata["embedding_model_revision"] == EMBEDDING_MODEL_REVISION
assert stage_3_metadata["sentence_transformers_version"] == "3.4.1"
assert stage_3_metadata["transformers_version"] == "4.49.0"
assert stage_3_metadata["dense_weight"] == DENSE_WEIGHT
assert stage_3_metadata["bm25_weight"] == BM25_WEIGHT
assert stage_3_metadata["rrf_constant"] == RRF_CONSTANT
assert stage_3_metadata["maximum_per_article"] == MAXIMUM_PER_ARTICLE
assert "Stage 3.1" in stage_3_metadata["retrieval_implementation_alignment"]

print("Clean chunks:", len(chunks))
print("Articles:", len({chunk["pmcid"] for chunk in chunks}))
print("Embedding matrix:", embeddings.shape)
print("Cleaned source records:", stage_3_metadata["cleaned_article_count"])

## 4. Hybrid retrieval

BM25 matches terms; dense retrieval matches meaning. Weighted Reciprocal Rank Fusion combines rankings, with a source-diversity limit. A retrieval score is not a grade out of five or a probability of correctness.

In [ ]:
STOP_WORDS = {
    "a", "an", "and", "are", "as", "at", "be", "been", "being", "by",
    "can", "could", "did", "do", "does", "for", "from", "had", "has",
    "have", "how", "in", "into", "is", "it", "its", "may", "of", "on",
    "or", "should", "that", "the", "their", "these", "this", "to", "use",
    "used", "using", "was", "were", "what", "when", "where", "which",
    "with", "would",
}
TOKEN_PATTERN = re.compile(r"[a-z0-9]+(?:-[a-z0-9]+)?")


def tokenize(text):
    """Create lower-case BM25 terms and remove common English words."""
    return [
        token
        for token in TOKEN_PATTERN.findall(text.lower())
        if token not in STOP_WORDS and len(token) >= 2
    ]


# The title and theme give BM25 useful document-level context.
document_tokens = [
    tokenize(f'{chunk["title"]} {chunk["theme"].replace("_", " ")} {chunk["text"]}')
    for chunk in chunks
]
document_lengths = np.array(
    [len(tokens) for tokens in document_tokens], dtype=np.float32
)
average_document_length = float(document_lengths.mean())
term_frequencies = [Counter(tokens) for tokens in document_tokens]

document_frequency = Counter()
for frequencies in term_frequencies:
    document_frequency.update(frequencies.keys())

document_count = len(chunks)
inverse_document_frequency = {
    term: math.log(1 + (document_count - frequency + 0.5) / (frequency + 0.5))
    for term, frequency in document_frequency.items()
}


def calculate_bm25_scores(question):
    """Calculate one BM25 relevance score for every chunk."""
    query_terms = tokenize(question)
    scores = np.zeros(document_count, dtype=np.float32)

    for document_index, frequencies in enumerate(term_frequencies):
        length_adjustment = BM25_K1 * (
            1 - BM25_B
            + BM25_B
            * document_lengths[document_index]
            / average_document_length
        )
        score = 0.0
        for term in query_terms:
            frequency = frequencies.get(term, 0)
            if frequency == 0:
                continue
            score += inverse_document_frequency.get(term, 0.0) * (
                frequency * (BM25_K1 + 1) / (frequency + length_adjustment)
            )
        scores[document_index] = score
    return scores


print("BM25 index created for", document_count, "chunks")

In [ ]:
# This small public model converts each new search query into a vector.
embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME, revision=EMBEDDING_MODEL_REVISION
)


def create_rank_positions(order):
    """Convert an ordered list of indexes into one-based rank positions."""
    positions = np.empty(len(order), dtype=np.int32)
    positions[order] = np.arange(1, len(order) + 1)
    return positions


def select_with_source_limit(order, maximum_per_article, top_k):
    """Return top results without allowing one source to dominate."""
    selected = []
    counts = Counter()
    for index in order:
        article_id = chunks[int(index)]["pmcid"]
        if counts[article_id] >= maximum_per_article:
            continue
        selected.append(int(index))
        counts[article_id] += 1
        if len(selected) == top_k:
            break
    return selected


def hybrid_retrieve(question, top_k=TOP_K_CANDIDATES):
    """Retrieve evidence using dense similarity and BM25 rank fusion."""
    question_embedding = embedding_model.encode(
        [question], convert_to_numpy=True, normalize_embeddings=True
    )[0]
    dense_scores = embeddings @ question_embedding
    bm25_scores = calculate_bm25_scores(question)

    dense_order = np.argsort(-dense_scores)
    bm25_order = np.argsort(-bm25_scores)
    dense_ranks = create_rank_positions(dense_order)
    bm25_ranks = create_rank_positions(bm25_order)

    hybrid_scores = (
        DENSE_WEIGHT / (RRF_CONSTANT + dense_ranks)
        + BM25_WEIGHT / (RRF_CONSTANT + bm25_ranks)
    )
    hybrid_order = np.argsort(-hybrid_scores)
    selected_indexes = select_with_source_limit(
        hybrid_order, MAXIMUM_PER_ARTICLE, top_k
    )

    results = []
    for rank, index in enumerate(selected_indexes, start=1):
        result = dict(chunks[index])
        result.update(
            {
                "rank": rank,
                "method": "hybrid_rrf",
                "dense_similarity": round(float(dense_scores[index]), 6),
                "bm25_score": round(float(bm25_scores[index]), 6),
                "hybrid_rrf_score": round(float(hybrid_scores[index]), 9),
            }
        )
        results.append(result)
    return results


# A short check confirms that retrieval still returns five different sources.
retrieval_check = hybrid_retrieve(
    "What hallucination risks occur when large language models write healthcare reports?"
)
assert len(retrieval_check) == TOP_K_CANDIDATES
assert len({item["pmcid"] for item in retrieval_check}) == TOP_K_CANDIDATES
print("Hybrid retriever check passed")

## 5. Load Qwen and the shared generation function

The downloaded open-weight model generates new text inside Colab. Four-bit loading saves GPU memory.

In [ ]:
# NF4 is a common 4-bit format for loading language models efficiently.
quantisation_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(
    GENERATOR_MODEL_NAME,
    revision=GENERATOR_MODEL_REVISION,
)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

language_model = AutoModelForCausalLM.from_pretrained(
    GENERATOR_MODEL_NAME,
    revision=GENERATOR_MODEL_REVISION,
    quantization_config=quantisation_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)
language_model.eval()

print("Free model loaded in 4-bit mode")

In [ ]:
def generate_text(system_instruction, user_instruction, max_new_tokens):
    """Send one instruction to the local model and return only its new text."""
    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": user_instruction},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer(prompt, return_tensors="pt")

    # Put inputs on the same device as the first model parameter.
    model_device = next(language_model.parameters()).device
    model_inputs = {key: value.to(model_device) for key, value in model_inputs.items()}
    input_length = model_inputs["input_ids"].shape[1]

    with torch.inference_mode():
        generated = language_model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = generated[0][input_length:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def extract_json_object(text):
    """Return the first valid JSON object, even if the model repeats text later."""
    decoder = json.JSONDecoder()
    for position, character in enumerate(text):
        if character != "{":
            continue
        try:
            value, _ = decoder.raw_decode(text[position:])
        except json.JSONDecodeError:
            continue
        if isinstance(value, dict):
            return value
    return None

## 6. Planner, evidence reviewer and writer

These Python functions give the shared model different jobs. The evidence reviewer can reject candidates; the writer is instructed to use selected evidence.

In [ ]:
def planner_agent(research_question):
    """Ask the model to create a structured report plan."""
    system_instruction = (
        "You are the planner in an academic research workflow. "
        "Plan a concise evidence-based report, not clinical advice. "
        "Return valid JSON only."
    )
    user_instruction = f"""
    Research question: {research_question}

    Return this structure:
    {{
      "report_title": "short title",
      "sections": [
        {{"heading": "section heading", "retrieval_query": "focused evidence query"}}
      ]
    }}

    Requirements:
    - exactly {MAX_REPORT_SECTIONS} sections;
    - include background/applications, benefits, risks/limitations, and conclusion;
    - each retrieval query must be understandable on its own;
    - do not include references because retrieval happens next.
    """
    raw_output = generate_text(
        system_instruction, user_instruction, PLANNER_MAX_TOKENS
    )
    plan = extract_json_object(raw_output)

    valid = (
        isinstance(plan, dict)
        and isinstance(plan.get("sections"), list)
        and len(plan["sections"]) == MAX_REPORT_SECTIONS
        and all(
            isinstance(section, dict)
            and section.get("heading")
            and section.get("retrieval_query")
            for section in plan["sections"]
        )
    )

    if not valid:
        # This fallback is visible in the saved trace and is not hidden.
        plan = {
            "report_title": "Opportunities and risks of large language models in healthcare",
            "sections": [
                {
                    "heading": "Background and healthcare applications",
                    "retrieval_query": "How are large language models used in healthcare research and clinical work?",
                },
                {
                    "heading": "Potential benefits",
                    "retrieval_query": "What benefits can large language models provide in healthcare reporting and decision support?",
                },
                {
                    "heading": "Factual, ethical and safety risks",
                    "retrieval_query": "What hallucination, misinformation, bias and safety risks affect healthcare use of large language models?",
                },
                {
                    "heading": "Safeguards and conclusion",
                    "retrieval_query": "What retrieval, citation and human oversight safeguards improve reliable use of language models in healthcare?",
                },
            ],
            "used_fallback": True,
        }
    else:
        plan["used_fallback"] = False

    return plan, raw_output

In [ ]:
def format_candidate_evidence(results):
    """Format retrieved passages for the evidence-reviewing prompt."""
    blocks = []
    for result in results:
        blocks.append(
            "\n".join(
                [
                    f'CHUNK_ID: {result["chunk_id"]}',
                    f'PMCID: {result["pmcid"]}',
                    f'TITLE: {result["title"]}',
                    f'THEME: {result["theme"]}',
                    f'PASSAGE: {result["text"]}',
                ]
            )
        )
    return "\n\n---\n\n".join(blocks)


def evidence_reviewer_agent(section_heading, retrieval_query, candidates):
    """Select passages that directly support one planned section."""
    # The development question is about LLMs. General medical-AI passages are
    # screened out before the model reviews the evidence.
    llm_markers = (
        "large language model",
        "language model",
        "llm",
        "chatgpt",
        "generative pre-trained transformer",
        "gpt",
    )
    screened_candidates = [
        item
        for item in candidates
        if item.get("theme") in DEVELOPMENT_EVIDENCE_THEMES
        and any(
            marker in (item.get("title", "") + " " + item.get("text", "")).lower()
            for marker in llm_markers
        )
    ]
    screening_fallback_used = len(screened_candidates) < 2
    if screening_fallback_used:
        screened_candidates = [
            item
            for item in candidates
            if any(
                marker in (item.get("title", "") + " " + item.get("text", "")).lower()
                for marker in llm_markers
            )
        ]
    if len(screened_candidates) < 2:
        screened_candidates = candidates

    system_instruction = (
        "You review evidence for an academic report. Select only passages "
        "that directly support the requested section and explicitly concern "
        "large language models or ChatGPT. Return valid JSON only."
    )
    user_instruction = f"""
    Section: {section_heading}
    Evidence question: {retrieval_query}

    Candidate passages:
    {format_candidate_evidence(screened_candidates)}

    Return:
    {{
      "selected_chunk_ids": ["chunk id", "chunk id"],
      "reason": "brief explanation",
      "limitations": "what the selected evidence does not establish"
    }}

    Select between 2 and 3 passages. Do not invent chunk IDs.
    """
    raw_output = generate_text(
        system_instruction, user_instruction, REVIEWER_MAX_TOKENS
    )
    decision = extract_json_object(raw_output)
    valid_ids = {item["chunk_id"] for item in screened_candidates}

    if isinstance(decision, dict):
        selected_ids = [
            chunk_id
            for chunk_id in decision.get("selected_chunk_ids", [])
            if chunk_id in valid_ids
        ][:3]
    else:
        selected_ids = []

    # Use the top three candidates if the model returns fewer than two valid IDs.
    fallback_used = len(selected_ids) < 2
    if fallback_used:
        selected_ids = [item["chunk_id"] for item in screened_candidates[:3]]

    selected = [
        item for item in screened_candidates if item["chunk_id"] in selected_ids
    ]
    return {
        "selected_evidence": selected,
        "raw_output": raw_output,
        "fallback_used": fallback_used,
        "screening_fallback_used": screening_fallback_used,
        "screened_candidate_ids": [
            item["chunk_id"] for item in screened_candidates
        ],
        "screened_out_chunk_ids": [
            item["chunk_id"]
            for item in candidates
            if item["chunk_id"] not in {
                screened_item["chunk_id"] for screened_item in screened_candidates
            }
        ],
        "reason": decision.get("reason", "") if isinstance(decision, dict) else "",
        "limitations": decision.get("limitations", "") if isinstance(decision, dict) else "",
    }

In [ ]:
def normalise_section_output(section_heading, generated_text):
    """Keep one complete section with the planned heading and no source list."""
    cleaned = generated_text.strip()
    cleaned = re.sub(r"^```(?:markdown)?\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    cleaned = cleaned.split("\n## Evidence sources", 1)[0].strip()

    expected_heading = f"## {section_heading}"
    if expected_heading in cleaned:
        cleaned = expected_heading + cleaned.split(expected_heading, 1)[1]
    else:
        # Replace an unexpected opening heading with the heading fixed by the plan.
        cleaned = re.sub(r"^#{1,3}\s+[^\n]+\n+", "", cleaned).strip()
        cleaned = expected_heading + "\n\n" + cleaned
    return cleaned.strip()


def normalise_generated_citations(section_text, selected_evidence):
    """Keep approved PMC labels and remove every other bracket label.

    An author-date label is never converted into a PMC identifier because
    matching words inside a passage cannot prove that the claim came from
    that passage. This prevents the normaliser from creating attribution.
    """
    allowed_pmcids = {item["pmcid"] for item in selected_evidence}
    changes = []

    def replace_label(match):
        label = match.group(1).strip()
        if re.fullmatch(r"PMC\d+", label) and label in allowed_pmcids:
            return match.group(0)
        changes.append(
            {
                "original_label": label,
                "replacement": "",
                "reason": (
                    "removed because it was not an already-approved PMC "
                    "identifier; the system does not invent a replacement"
                ),
            }
        )
        return ""

    cleaned = re.sub(r"\[([^\[\]\n]+)\]", replace_label, section_text)
    cleaned = re.sub(r"[ \t]+([.,;:])", r"\1", cleaned)
    return cleaned, changes


def calculate_section_checks(section_text, selected_evidence):
    """Require one cited paragraph per passage and a citation per sentence."""
    allowed_pmcids = {item["pmcid"] for item in selected_evidence}
    labels = re.findall(r"\[([^\[\]\n]+)\]", section_text)
    invalid_labels = sorted(
        {
            label
            for label in labels
            if not re.fullmatch(r"PMC\d+", label)
            or label not in allowed_pmcids
        }
    )
    paragraphs = [
        paragraph.strip()
        for paragraph in re.split(r"\n\s*\n", section_text)
        if paragraph.strip() and not paragraph.strip().startswith("#")
    ]
    substantive = [paragraph for paragraph in paragraphs if len(paragraph.split()) >= 25]
    # Standard punctuation may appear after the final citation.
    ending_pattern = r"(?:\s*\[PMC\d+\])+(?:[.!?])?\s*$"
    ending_with_allowed_citation = [
        paragraph
        for paragraph in substantive
        if re.search(ending_pattern, paragraph)
        and all(pmcid in allowed_pmcids for pmcid in re.findall(r"\[(PMC\d+)\]", paragraph))
    ]
    substantive_sentences = []
    for paragraph in substantive:
        substantive_sentences.extend(
            sentence.strip()
            for sentence in re.split(
                r"(?<=[.!?])\s+(?=[A-Z0-9])", paragraph
            )
            if len(re.sub(r"\[PMC\d+\]", "", sentence).split()) >= 6
        )
    cited_substantive_sentences = [
        sentence
        for sentence in substantive_sentences
        if re.search(r"\[PMC\d+\]", sentence)
    ]
    expected_paragraph_count = len(selected_evidence)
    complete_ending = section_text.strip().endswith((".", "!", "?"))
    return {
        "invalid_citations": invalid_labels,
        "expected_paragraph_count": expected_paragraph_count,
        "substantive_paragraph_count": len(substantive),
        "paragraphs_ending_with_allowed_citation_count": len(ending_with_allowed_citation),
        "substantive_sentence_count": len(substantive_sentences),
        "cited_substantive_sentence_count": len(cited_substantive_sentences),
        "citation_paragraph_coverage": round(
            len(ending_with_allowed_citation) / len(substantive), 3
        ) if substantive else 0.0,
        "citation_sentence_coverage": round(
            len(cited_substantive_sentences) / len(substantive_sentences), 3
        ) if substantive_sentences else 0.0,
        "complete_ending": complete_ending,
        "section_gate_passed": (
            not invalid_labels
            and expected_paragraph_count in (2, 3)
            and len(substantive) == expected_paragraph_count
            and len(ending_with_allowed_citation) == expected_paragraph_count
            and len(cited_substantive_sentences) == len(substantive_sentences)
            and len(substantive_sentences) >= expected_paragraph_count
            and complete_ending
        ),
    }


def prepare_passage_bound_paragraph(raw_text, pmcid):
    """Clean one passage-specific paraphrase and attach its known PMCID."""
    cleaned = raw_text.strip()
    cleaned = re.sub(r"^```(?:markdown)?\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    cleaned = re.sub(r"^#{1,3}\s+[^\n]+\n*", "", cleaned).strip()
    cleaned = re.sub(r"\[[^\[\]\n]+\]", "", cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    sentences = [
        sentence.strip()
        for sentence in re.split(r"(?<=[.!?])\s+(?=[A-Z0-9])", cleaned)
        if len(sentence.split()) >= 6
    ][:3]
    cited_sentences = []
    for sentence in sentences:
        sentence_core = re.sub(r"[.!?]+$", "", sentence).strip()
        cited_sentences.append(f"{sentence_core} [{pmcid}].")
    return " ".join(cited_sentences)


def writer_agent(research_question, section_heading, selected_evidence):
    """Write one independently traceable paragraph per evidence passage."""
    paragraphs = []
    passage_traces = []

    for evidence_item in selected_evidence:
        system_instruction = (
            "You paraphrase one supplied academic passage. Use only facts "
            "explicitly stated in that passage. Do not infer, combine outside "
            "knowledge, mention citations or provide medical advice."
        )
        user_instruction = f"""
        Overall question: {research_question}
        Section: {section_heading}

        One approved passage:
        {evidence_item["text"]}

        Write one coherent paragraph of 45 to 70 words containing two or
        three concise factual sentences. Every sentence must be a close
        paraphrase of this passage. Return only the paragraph, without a
        heading, citation, bullet, author name or publication year.
        """
        first_raw = generate_text(system_instruction, user_instruction, 140)
        paragraph = prepare_passage_bound_paragraph(
            first_raw, evidence_item["pmcid"]
        )
        retry_raw = None

        # Retry once only when the first response is incomplete.
        if len(paragraph.split()) < 25 or not paragraph.endswith("."):
            retry_instruction = user_instruction + (
                "\nYour first response was too short or incomplete. Rewrite it "
                "as 45 to 70 words and finish every sentence."
            )
            retry_raw = generate_text(system_instruction, retry_instruction, 140)
            retry_paragraph = prepare_passage_bound_paragraph(
                retry_raw, evidence_item["pmcid"]
            )
            if len(retry_paragraph.split()) >= len(paragraph.split()):
                paragraph = retry_paragraph

        paragraphs.append(paragraph)
        passage_traces.append(
            {
                "chunk_id": evidence_item["chunk_id"],
                "pmcid": evidence_item["pmcid"],
                "first_raw_output": first_raw,
                "retry_raw_output": retry_raw,
                "final_paragraph": paragraph,
                "citation_mapping": (
                    "Generated from this passage alone; its known PMCID was "
                    "attached deterministically to each sentence."
                ),
            }
        )

    section_text = (
        f"## {section_heading}\n\n" + "\n\n".join(paragraphs)
    ).strip()
    checks = calculate_section_checks(section_text, selected_evidence)
    return {
        "section_text": section_text,
        "raw_output": json.dumps(passage_traces, ensure_ascii=False),
        "repair_raw_output": None,
        "citation_normalisations": [
            {
                "chunk_id": trace["chunk_id"],
                "pmcid": trace["pmcid"],
                "reason": trace["citation_mapping"],
            }
            for trace in passage_traces
        ],
        "section_checks": checks,
    }


## 7. Structural checks and orchestration

The controller records intermediate outputs and applies conditional corrections. Failed checks remain visible; a citation alone is not proof of factual support.

In [ ]:
def calculate_report_checks(report_text, allowed_pmcids):
    """Calculate transparent citation and completeness checks without an LLM."""
    # The source list is metadata, so only the generated report body is assessed.
    report_body = report_text.split("\n\n## Evidence sources", 1)[0].strip()
    allowed_pmcids = set(allowed_pmcids)

    # Inspect every square-bracket citation, not only labels that already look valid.
    citation_labels = re.findall(r"\[([^\[\]\n]+)\]", report_body)
    cited_pmcids = [
        label for label in citation_labels if re.fullmatch(r"PMC\d+", label)
    ]
    unique_citations = sorted(set(cited_pmcids))
    invalid_citations = sorted(
        {
            label
            for label in citation_labels
            if not re.fullmatch(r"PMC\d+", label)
            or label not in allowed_pmcids
        }
    )

    # A substantive paragraph has at least 25 words and is not a heading.
    paragraphs = [
        paragraph.strip()
        for paragraph in re.split(r"\n\s*\n", report_body)
        if paragraph.strip()
    ]
    substantive_paragraphs = [
        paragraph
        for paragraph in paragraphs
        if not paragraph.startswith("#") and len(paragraph.split()) >= 25
    ]
    cited_paragraphs = [
        paragraph
        for paragraph in substantive_paragraphs
        if re.search(r"(?:\s*\[PMC\d+\])+(?:[.!?])?\s*$", paragraph)
    ]
    coverage = (
        len(cited_paragraphs) / len(substantive_paragraphs)
        if substantive_paragraphs
        else 0.0
    )
    section_headings = re.findall(r"^##\s+(.+)$", report_body, flags=re.MULTILINE)
    body_without_trailing_citations = re.sub(
        r"(?:\s*\[PMC\d+\])+\s*$", "", report_body
    ).rstrip()
    body_ends_with_punctuation = body_without_trailing_citations.endswith(
        (".", "!", "?")
    )
    deterministic_pass = (
        not invalid_citations
        and coverage == 1.0
        and MAX_REPORT_SECTIONS * 2 <= len(substantive_paragraphs) <= MAX_REPORT_SECTIONS * 3
        and len(section_headings) == MAX_REPORT_SECTIONS
        and body_ends_with_punctuation
    )

    return {
        "word_count": len(report_body.split()),
        "section_count": len(section_headings),
        "section_headings": section_headings,
        "body_ends_with_punctuation": body_ends_with_punctuation,
        "unique_citation_count": len(unique_citations),
        "unique_citations": unique_citations,
        "invalid_citations": invalid_citations,
        "substantive_paragraph_count": len(substantive_paragraphs),
        "paragraphs_ending_with_citation_count": len(cited_paragraphs),
        "citation_paragraph_coverage": round(coverage, 3),
        "structural_gate_passed": deterministic_pass,
    }


def report_reviewer_agent(research_question, report_text, evidence):
    """Review the complete report and return structured development feedback."""
    allowed_pmcids = sorted({item["pmcid"] for item in evidence})
    checks = calculate_report_checks(report_text, allowed_pmcids)

    # Short evidence summaries keep the review prompt within the free model limit.
    evidence_summary = "\n\n".join(
        f'{item["pmcid"]} | {item["title"]}\n{item["text"][:700]}'
        for item in evidence
    )
    system_instruction = (
        "You are the final reviewer in an academic RAG workflow. Evaluate the "
        "report only against the supplied evidence. Return valid JSON only."
    )
    user_instruction = f"""
    Research question: {research_question}

    Report:
    {report_text}

    Retrieved evidence:
    {evidence_summary}

    Deterministic checks:
    {json.dumps(checks, indent=2)}

    Score each criterion from 1 (unsupported or poor) to 5 (strongly supported).
    A score of 3 means adequate but with clear limitations. Return exactly one JSON object:
    {{
      "factual_support": 1,
      "relevance": 1,
      "coherence": 1,
      "completeness": 1,
      "citation_quality": 1,
      "needs_revision": true,
      "feedback": ["specific improvement"]
    }}

    Set needs_revision to true if any score is below 3 or if a deterministic check fails.
    Do not write any text after the JSON object.
    """
    raw_output = generate_text(
        system_instruction, user_instruction, REPORT_REVIEW_MAX_TOKENS
    )
    review = extract_json_object(raw_output)
    score_names = [
        "factual_support",
        "relevance",
        "coherence",
        "completeness",
        "citation_quality",
    ]
    review_format_valid = (
        isinstance(review, dict)
        and all(
            isinstance(review.get(score_name), (int, float))
            and 1 <= review[score_name] <= 5
            for score_name in score_names
        )
        and isinstance(review.get("needs_revision"), bool)
    )

    if not review_format_valid:
        review = {
            "needs_revision": True,
            "feedback": ["The reviewer did not return valid JSON; inspect the raw output."],
            "parse_fallback_used": True,
        }
    else:
        review["parse_fallback_used"] = False

    if not isinstance(review.get("feedback"), list):
        review["feedback"] = [str(review.get("feedback", "Review the report."))]

    deterministic_failures = []
    if checks["invalid_citations"]:
        deterministic_failures.append(
            "Remove invalid citations: " + ", ".join(checks["invalid_citations"])
        )
    if checks["citation_paragraph_coverage"] < 1.0:
        deterministic_failures.append(
            "Make every substantive paragraph end with an approved PMC citation."
        )
    if not (MAX_REPORT_SECTIONS * 2 <= checks["substantive_paragraph_count"] <= MAX_REPORT_SECTIONS * 3):
        deterministic_failures.append(
            "Return two or three substantive paragraphs in each section."
        )
    if checks["section_count"] != MAX_REPORT_SECTIONS:
        deterministic_failures.append(
            f"Return exactly {MAX_REPORT_SECTIONS} report sections."
        )
    if not checks["body_ends_with_punctuation"]:
        deterministic_failures.append("Complete the final sentence of the report.")

    score_failures = [
        score_name
        for score_name in score_names
        if isinstance(review.get(score_name), (int, float))
        and review[score_name] < 3
    ]
    if deterministic_failures or score_failures:
        review["needs_revision"] = True
    for failure in deterministic_failures:
        if failure not in review["feedback"]:
            review["feedback"].append(failure)

    review["deterministic_checks"] = checks
    review["raw_output"] = raw_output
    return review

In [ ]:
def build_evidence_source_list(evidence):
    """Create one source-list entry per PMCID, even if two chunks were used."""
    source_by_pmcid = {}
    for item in evidence:
        source_by_pmcid.setdefault(item["pmcid"], item)
    return "\n".join(
        f'- [{item["pmcid"]}] {item["title"]}. {item["source_url"]}'
        for item in source_by_pmcid.values()
    )


def revision_agent(
    research_question,
    section_heading,
    original_section,
    selected_evidence,
    review_feedback,
):
    """Revise one section using only the evidence originally approved for it."""
    evidence_text = format_candidate_evidence(selected_evidence)
    allowed_citations = sorted({item["pmcid"] for item in selected_evidence})
    system_instruction = (
        "You revise one section in an academic RAG workflow. Use only the supplied "
        "PMC evidence and identifiers. Never copy author-date citations from inside "
        "a passage. Do not add outside facts, statistics or references. Return only "
        "the revised section."
    )
    user_instruction = f"""
    Overall research question: {research_question}
    Section heading: {section_heading}
    Allowed PMC citations: {allowed_citations}
    General reviewer feedback: {json.dumps(review_feedback)}

    Approved evidence for this section:
    {evidence_text}

    Original section:
    {original_section}

    Revise the section in 120 to 180 words and exactly {len(selected_evidence)} substantive paragraphs.
    Requirements:
    - start with exactly: ## {section_heading}
    - every paragraph must end with at least one allowed PMC citation;
    - use only the allowed PMC identifiers in square brackets;
    - do not mention authors, publication years or citations found inside a passage;
    - preserve uncertainty and avoid clinical advice;
    - finish the final sentence;
    - do not add a source list or any text after the section.
    """
    raw_output = generate_text(
        system_instruction, user_instruction, WRITER_MAX_TOKENS
    )
    candidate_section = normalise_section_output(section_heading, raw_output)
    candidate_section, citation_changes = normalise_generated_citations(
        candidate_section, selected_evidence
    )
    checks = calculate_section_checks(candidate_section, selected_evidence)
    revision_accepted = checks["section_gate_passed"]

    # A revision is never allowed to replace a previously valid section with
    # one that fails the deterministic section gate.
    final_section = candidate_section if revision_accepted else original_section
    return {
        "section_text": final_section,
        "raw_output": raw_output,
        "candidate_section": candidate_section,
        "citation_normalisations": citation_changes,
        "section_checks": checks,
        "revision_accepted": revision_accepted,
    }


def orchestrate_report(research_question):
    """Run the complete planner-retriever-reviewer-writer workflow."""
    started_at = datetime.now(timezone.utc)
    plan, planner_raw_output = planner_agent(research_question)

    section_traces = []
    written_sections = []
    all_selected_evidence = []

    for section_number, section in enumerate(plan["sections"], start=1):
        print(f'Working on section {section_number}/{len(plan["sections"])}: {section["heading"]}')

        candidates = hybrid_retrieve(section["retrieval_query"])
        evidence_decision = evidence_reviewer_agent(
            section["heading"], section["retrieval_query"], candidates
        )
        selected_evidence = evidence_decision["selected_evidence"]
        writer_result = writer_agent(
            research_question, section["heading"], selected_evidence
        )
        section_text = writer_result["section_text"]

        written_sections.append(section_text)
        all_selected_evidence.extend(selected_evidence)
        section_traces.append(
            {
                "section_number": section_number,
                "heading": section["heading"],
                "retrieval_query": section["retrieval_query"],
                "retrieved_candidates": candidates,
                "evidence_review": {
                    key: value
                    for key, value in evidence_decision.items()
                    if key != "selected_evidence"
                },
                "selected_chunk_ids": [
                    item["chunk_id"] for item in selected_evidence
                ],
                "written_section": section_text,
                "writer_raw_output": writer_result["raw_output"],
                "writer_repair_raw_output": writer_result[
                    "repair_raw_output"
                ],
                "writer_citation_normalisations": writer_result[
                    "citation_normalisations"
                ],
                "writer_section_checks": writer_result["section_checks"],
            }
        )

    # Remove repeated chunks while keeping their first appearance.
    evidence_by_id = {}
    for item in all_selected_evidence:
        evidence_by_id.setdefault(item["chunk_id"], item)
    unique_evidence = list(evidence_by_id.values())

    evidence_source_list = build_evidence_source_list(unique_evidence)
    report_text = (
        f'# {plan["report_title"]}\n\n'
        + "\n\n".join(written_sections)
        + "\n\n## Evidence sources\n\n"
        + evidence_source_list
    )

    initial_review = report_reviewer_agent(
        research_question, report_text, unique_evidence
    )
    revision_performed = bool(initial_review.get("needs_revision", False))

    if revision_performed:
        evidence_by_chunk = {
            item["chunk_id"]: item for item in unique_evidence
        }
        revised_sections = []
        for trace in section_traces:
            section_evidence = [
                evidence_by_chunk[chunk_id]
                for chunk_id in trace["selected_chunk_ids"]
                if chunk_id in evidence_by_chunk
            ]
            revision_result = revision_agent(
                research_question,
                trace["heading"],
                trace["written_section"],
                section_evidence,
                initial_review.get("feedback", []),
            )
            trace["revised_section"] = revision_result["section_text"]
            trace["revision_raw_output"] = revision_result["raw_output"]
            trace["revision_candidate_section"] = revision_result[
                "candidate_section"
            ]
            trace["revision_citation_normalisations"] = revision_result[
                "citation_normalisations"
            ]
            trace["revision_section_checks"] = revision_result[
                "section_checks"
            ]
            trace["revision_accepted"] = revision_result["revision_accepted"]
            revised_sections.append(revision_result["section_text"])

        # Rebuild the report from four complete sections and the deduplicated source list.
        report_text = (
            f'# {plan["report_title"]}\n\n'
            + "\n\n".join(revised_sections)
            + "\n\n## Evidence sources\n\n"
            + evidence_source_list
        )
        final_review = report_reviewer_agent(
            research_question, report_text, unique_evidence
        )
    else:
        final_review = initial_review

    finished_at = datetime.now(timezone.utc)
    return {
        "research_question": research_question,
        "plan": plan,
        "planner_raw_output": planner_raw_output,
        "section_traces": section_traces,
        "selected_evidence": unique_evidence,
        "initial_review": initial_review,
        "revision_performed": revision_performed,
        "final_review": final_review,
        "structural_gate_passed": bool(
            final_review["deterministic_checks"][
                "structural_gate_passed"
            ]
        ),
        "internal_reviewer_requests_further_revision": bool(
            final_review.get("needs_revision", True)
        ),
        "report_text": report_text,
        "started_utc": started_at.isoformat(),
        "finished_utc": finished_at.isoformat(),
        "duration_seconds": round((finished_at - started_at).total_seconds(), 2),
    }

## 8. Load the separate claim verifier

NLI compares each claim with cited evidence. Its supported/neutral/contradicted labels are automated assessments, not human judgements.

In [ ]:
from transformers import AutoModelForSequenceClassification

# This separate model checks evidence/claim sentence pairs.  The exact
# revision is pinned so that a later run loads the same model files.
NLI_MODEL_NAME = "cross-encoder/nli-deberta-v3-base"
NLI_MODEL_REVISION = "6c749ce3425cd33b46d187e45b92bbf96ee12ec7"
NLI_ENTAILMENT_THRESHOLD = 0.50
CLAIM_SUPPORT_GATE = 0.80
NLI_BATCH_SIZE = 8

nli_device = "cuda" if torch.cuda.is_available() else "cpu"
nli_tokenizer = AutoTokenizer.from_pretrained(
    NLI_MODEL_NAME,
    revision=NLI_MODEL_REVISION,
)
nli_model = AutoModelForSequenceClassification.from_pretrained(
    NLI_MODEL_NAME,
    revision=NLI_MODEL_REVISION,
    torch_dtype=torch.float16 if nli_device == "cuda" else torch.float32,
).to(nli_device)
nli_model.eval()
nli_labels = {
    str(label).lower() for label in nli_model.config.id2label.values()
}
assert {"entailment", "contradiction", "neutral"}.issubset(nli_labels)

print("Claim verifier loaded:", NLI_MODEL_NAME)
print("Verifier device:", nli_device)
print("Entailment threshold:", NLI_ENTAILMENT_THRESHOLD)

In [ ]:
def split_report_sections(report_text):
    """Return the generated sections without the title or source list."""
    body = report_text.split("\n\n## Evidence sources", 1)[0].strip()
    heading_matches = list(re.finditer(r"^##\s+(.+)$", body, flags=re.MULTILINE))
    sections = {}
    for index, match in enumerate(heading_matches):
        start = match.start()
        end = (
            heading_matches[index + 1].start()
            if index + 1 < len(heading_matches)
            else len(body)
        )
        sections[match.group(1).strip()] = body[start:end].strip()
    return sections


def split_substantive_sentences(section_text):
    """Split one section while keeping inline citations with their claims."""
    body = re.sub(r"^##\s+[^\n]+\n*", "", section_text.strip())
    sentences = []
    for paragraph_number, paragraph in enumerate(
        re.split(r"\n\s*\n", body), start=1
    ):
        compact = re.sub(r"\s+", " ", paragraph).strip()
        if not compact:
            continue
        parts = re.split(
            r"(?<=[.!?])\s+(?=(?:\[PMC\d+\]\s*)*[A-Z0-9])",
            compact,
        )
        for sentence_number, sentence in enumerate(parts, start=1):
            sentence = sentence.strip()
            claim_text = re.sub(r"\[PMC\d+\]", "", sentence)
            claim_text = re.sub(r"\s+", " ", claim_text).strip()
            if len(claim_text.split()) >= 6:
                sentences.append(
                    {
                        "paragraph_number": paragraph_number,
                        "sentence_number": sentence_number,
                        "sentence": sentence,
                        "claim": claim_text,
                    }
                )
    return sentences


def calculate_claim_metrics(rows):
    """Summarise transparent claim-level grounding outcomes."""
    total = len(rows)
    counts = Counter(row["status"] for row in rows)
    supported = counts.get("supported", 0)
    support_rate = supported / total if total else 0.0
    return {
        "total_claims": total,
        "supported_claims": supported,
        "unsupported_claims": counts.get("unsupported", 0),
        "contradicted_claims": counts.get("contradicted", 0),
        "missing_citation_claims": counts.get("missing_citation", 0),
        "invalid_citation_claims": counts.get("invalid_citation", 0),
        "claim_support_rate": round(support_rate, 3),
        "claim_grounding_gate_passed": (
            support_rate >= CLAIM_SUPPORT_GATE
            and counts.get("contradicted", 0) == 0
            and counts.get("missing_citation", 0) == 0
            and counts.get("invalid_citation", 0) == 0
        ),
    }


def build_nli_evidence_windows(passage_text):
    """Create short one- and two-sentence premises from a long passage."""
    compact = re.sub(r"\s+", " ", passage_text).strip()
    sentences = [
        sentence.strip()
        for sentence in re.split(r"(?<=[.!?])\s+(?=[A-Z0-9])", compact)
        if len(sentence.split()) >= 5
    ]
    windows = list(sentences)
    windows.extend(
        sentences[index] + " " + sentences[index + 1]
        for index in range(len(sentences) - 1)
    )
    # Retain order while removing duplicate windows.
    return list(dict.fromkeys(windows)) or [compact]


def audit_report_claims(report_text, selected_evidence):
    """Score each report sentence against short windows from its cited passage."""
    evidence_by_pmcid = {}
    for item in selected_evidence:
        evidence_by_pmcid.setdefault(item["pmcid"], []).append(item)

    rows = []
    pending_pairs = []
    for heading, section_text in split_report_sections(report_text).items():
        for sentence_data in split_substantive_sentences(section_text):
            citations = sorted(
                set(re.findall(r"\[(PMC\d+)\]", sentence_data["sentence"]))
            )
            row = {
                "section": heading,
                **sentence_data,
                "citations": citations,
                "status": "pending",
                "best_pmcid": "",
                "best_chunk_id": "",
                "best_evidence_window": "",
                "nli_label": "",
                "entailment_probability": 0.0,
                "contradiction_probability": 0.0,
                "neutral_probability": 0.0,
            }
            row_index = len(rows)
            rows.append(row)

            if not citations:
                row["status"] = "missing_citation"
                continue
            if any(pmcid not in evidence_by_pmcid for pmcid in citations):
                row["status"] = "invalid_citation"
                continue

            # Short evidence windows make the verifier compare the claim with
            # the relevant sentence rather than an unrelated part of a long chunk.
            for pmcid in citations:
                for evidence_item in evidence_by_pmcid[pmcid]:
                    for evidence_window in build_nli_evidence_windows(
                        evidence_item["text"]
                    ):
                        pending_pairs.append(
                            {
                                "row_index": row_index,
                                "pmcid": pmcid,
                                "chunk_id": evidence_item["chunk_id"],
                                "premise": evidence_window,
                                "hypothesis": sentence_data["claim"],
                            }
                        )

    pair_results = []
    for start in range(0, len(pending_pairs), NLI_BATCH_SIZE):
        batch = pending_pairs[start : start + NLI_BATCH_SIZE]
        encoded = nli_tokenizer(
            [item["premise"] for item in batch],
            [item["hypothesis"] for item in batch],
            padding=True,
            truncation="only_first",
            max_length=512,
            return_tensors="pt",
        ).to(nli_device)
        with torch.no_grad():
            logits = nli_model(**encoded).logits
            probabilities = torch.softmax(logits.float(), dim=-1).cpu().numpy()

        for item, scores in zip(batch, probabilities):
            label_scores = {
                str(nli_model.config.id2label[index]).lower(): float(score)
                for index, score in enumerate(scores)
            }
            pair_results.append({**item, **label_scores})

    results_by_row = {}
    for result in pair_results:
        results_by_row.setdefault(result["row_index"], []).append(result)

    for row_index, candidates in results_by_row.items():
        best = max(candidates, key=lambda item: item.get("entailment", 0.0))
        row = rows[row_index]
        row["best_pmcid"] = best["pmcid"]
        row["best_chunk_id"] = best["chunk_id"]
        row["best_evidence_window"] = best["premise"]
        row["entailment_probability"] = round(best.get("entailment", 0.0), 4)
        row["contradiction_probability"] = round(
            best.get("contradiction", 0.0), 4
        )
        row["neutral_probability"] = round(best.get("neutral", 0.0), 4)
        label_probabilities = {
            "contradiction": best.get("contradiction", 0.0),
            "entailment": best.get("entailment", 0.0),
            "neutral": best.get("neutral", 0.0),
        }
        row["nli_label"] = max(label_probabilities, key=label_probabilities.get)
        if (
            row["nli_label"] == "entailment"
            and row["entailment_probability"] >= NLI_ENTAILMENT_THRESHOLD
        ):
            row["status"] = "supported"
        elif row["nli_label"] == "contradiction":
            row["status"] = "contradicted"
        else:
            row["status"] = "unsupported"

    return {
        "rows": rows,
        "metrics": calculate_claim_metrics(rows),
    }

## 9. Optional A/B baselines and the Condition C wrapper

A is direct generation without retrieval. B uses agent roles without retrieval. C uses the retrieval-and-verification workflow. Only C runs by default.

In [ ]:
def clean_generated_report(raw_text):
    """Remove code fences without changing the report content."""
    cleaned = raw_text.strip()
    cleaned = re.sub(
        r"^```(?:markdown)?\s*", "", cleaned, flags=re.IGNORECASE
    )
    cleaned = re.sub(r"\s*```$", "", cleaned)
    return cleaned.strip()


def normalise_no_rag_heading_levels(report_text, planned_headings=None):
    """Standardise Markdown levels without changing report wording."""
    lines = report_text.splitlines()
    parsed_headings = []
    for line_number, line in enumerate(lines):
        match = re.match(r"^(#{1,6})\s+(.+?)\s*$", line)
        if match:
            parsed_headings.append(
                {
                    "line_number": line_number,
                    "level": len(match.group(1)),
                    "heading": match.group(2).strip(),
                }
            )

    changes = []
    if planned_headings:
        # The agent plan identifies the four section names for Condition B.
        planned_lookup = {
            re.sub(r"[^a-z0-9]+", " ", heading.lower()).strip(): heading
            for heading in planned_headings
        }
        for item in parsed_headings:
            key = re.sub(
                r"[^a-z0-9]+", " ", item["heading"].lower()
            ).strip()
            if key in planned_lookup:
                replacement = "## " + planned_lookup[key]
                if lines[item["line_number"]] != replacement:
                    changes.append(
                        {
                            "original": lines[item["line_number"]],
                            "replacement": replacement,
                            "reason": "matched a section fixed by the agent plan",
                        }
                    )
                    lines[item["line_number"]] = replacement
    else:
        # Condition A has no plan. Select the heading depth containing
        # exactly four major headings; deeper subheadings are untouched.
        level_counts = Counter(
            item["level"] for item in parsed_headings
        )
        candidate_levels = sorted(
            level for level, count in level_counts.items() if count == 4
        )
        target_level = candidate_levels[0] if candidate_levels else None
        if target_level is not None:
            for item in parsed_headings:
                if item["level"] == target_level:
                    replacement = "## " + item["heading"]
                    if lines[item["line_number"]] != replacement:
                        changes.append(
                            {
                                "original": lines[item["line_number"]],
                                "replacement": replacement,
                                "reason": (
                                    "identified as one of four major sections"
                                ),
                            }
                        )
                        lines[item["line_number"]] = replacement

    return "\n".join(lines).strip(), changes


def no_rag_report_checks(report_text):
    """Measure comparable completeness for conditions without RAG."""
    headings = re.findall(
        r"^##\s+(.+)$", report_text, flags=re.MULTILINE
    )
    word_count = len(report_text.split())
    complete = report_text.rstrip().endswith((".", "!", "?"))
    return {
        "word_count": word_count,
        "section_count": len(headings),
        "section_headings": headings,
        "complete_ending": complete,
        "target_length_passed": 300 <= word_count <= 500,
        "basic_structure_passed": len(headings) == 4 and complete,
    }


def run_condition_a(question):
    """Generate one report without planning, retrieval or revision."""
    started = datetime.now(timezone.utc)
    system_instruction = (
        "Write a concise academic research report. This is a research "
        "demonstration, not clinical advice. Do not invent references."
    )
    user_instruction = f"""
    Research question: {question}

    Write a 350 to 450 word report with exactly four level-two Markdown
    sections. Address the question, benefits, limitations and a balanced
    conclusion. Use internal model knowledge only. Do not use citations or
    claim that evidence was retrieved. Return only the report.
    """
    raw_output = generate_text(
        system_instruction, user_instruction, 620
    )
    cleaned_report = clean_generated_report(raw_output)
    report_text, heading_changes = normalise_no_rag_heading_levels(
        cleaned_report
    )
    finished = datetime.now(timezone.utc)
    return {
        "condition": "A_single_pass_no_RAG",
        "question": question,
        "report_text": report_text,
        "raw_output": raw_output,
        "heading_level_normalisations": heading_changes,
        "checks": no_rag_report_checks(report_text),
        "started_utc": started.isoformat(),
        "finished_utc": finished.isoformat(),
        "duration_seconds": round(
            (finished - started).total_seconds(), 2
        ),
    }

In [ ]:
def no_rag_planner_agent(question):
    """Create a report plan without requesting or receiving evidence."""
    system_instruction = (
        "You are the planner in an academic report workflow. Plan a "
        "balanced report using internal model knowledge only. Return JSON only."
    )
    user_instruction = f"""
    Research question: {question}

    Return exactly:
    {{
      "report_title": "short title",
      "sections": [
        {{"heading": "section heading", "purpose": "section purpose"}}
      ]
    }}

    Include exactly four sections covering context and applications,
    opportunities, risks and limitations, and a balanced conclusion.
    Do not include citations, references or retrieval instructions.
    """
    raw_output = generate_text(
        system_instruction, user_instruction, PLANNER_MAX_TOKENS
    )
    plan = extract_json_object(raw_output)
    valid = (
        isinstance(plan, dict)
        and isinstance(plan.get("report_title"), str)
        and isinstance(plan.get("sections"), list)
        and len(plan["sections"]) == MAX_REPORT_SECTIONS
        and all(
            isinstance(section, dict)
            and section.get("heading")
            and section.get("purpose")
            for section in plan["sections"]
        )
    )
    if not valid:
        plan = {
            "report_title": "Large language models in healthcare reporting",
            "sections": [
                {
                    "heading": "Context and applications",
                    "purpose": "Explain relevant healthcare report uses.",
                },
                {
                    "heading": "Potential opportunities",
                    "purpose": "Discuss potential research and reporting benefits.",
                },
                {
                    "heading": "Risks and limitations",
                    "purpose": "Discuss factual, ethical and practical risks.",
                },
                {
                    "heading": "Balanced conclusion",
                    "purpose": "Summarise careful and responsible use.",
                },
            ],
            "parse_fallback_used": True,
        }
    else:
        plan["parse_fallback_used"] = False
    return plan, raw_output


def no_rag_writer_agent(question, plan):
    """Write from an agent plan without retrieved evidence."""
    headings = [section["heading"] for section in plan["sections"]]
    system_instruction = (
        "You are the writer in an agentic academic workflow. Follow the "
        "plan using internal model knowledge only. Do not invent citations, "
        "sources or statistics. This is not clinical advice."
    )
    user_instruction = f"""
    Research question: {question}
    Planned title: {plan["report_title"]}
    Required headings: {headings}

    Write a 350 to 450 word report. Start with the title as one level-one
    heading and use exactly the four required level-two headings in order.
    Use no citations or reference list. Return only the report.
    """
    raw_output = generate_text(system_instruction, user_instruction, 650)
    cleaned_report = clean_generated_report(raw_output)
    report_text, heading_changes = normalise_no_rag_heading_levels(
        cleaned_report, headings
    )
    return report_text, raw_output, heading_changes


def no_rag_reviewer_agent(question, report_text):
    """Review quality without pretending to verify external facts."""
    system_instruction = (
        "Review relevance, coherence and completeness. You have no retrieved "
        "sources, so do not claim to verify factual accuracy. Return JSON only."
    )
    user_instruction = f"""
    Research question: {question}
    Report: {report_text}

    Return exactly:
    {{
      "relevance": 1,
      "coherence": 1,
      "completeness": 1,
      "needs_revision": true,
      "feedback": ["specific improvement"]
    }}
    Score 1 to 5 and request revision if any score is below 3.
    """
    raw_output = generate_text(system_instruction, user_instruction, 260)
    review = extract_json_object(raw_output)
    score_names = ["relevance", "coherence", "completeness"]
    valid = (
        isinstance(review, dict)
        and all(
            isinstance(review.get(name), (int, float))
            and 1 <= review[name] <= 5
            for name in score_names
        )
        and isinstance(review.get("needs_revision"), bool)
    )
    if not valid:
        review = {
            "relevance": None,
            "coherence": None,
            "completeness": None,
            "needs_revision": True,
            "feedback": ["Invalid reviewer JSON; inspect the raw output."],
            "parse_fallback_used": True,
        }
    else:
        review["parse_fallback_used"] = False
    review["raw_output"] = raw_output
    return review


def run_condition_b(question):
    """Run planner, writer and reviewer roles without retrieval."""
    started = datetime.now(timezone.utc)
    plan, planner_raw = no_rag_planner_agent(question)
    initial_report, writer_raw, initial_heading_changes = no_rag_writer_agent(
        question, plan
    )
    review = no_rag_reviewer_agent(question, initial_report)
    final_report = initial_report
    revision_raw = None
    revision_accepted = False

    if review.get("needs_revision", True):
        system_instruction = (
            "Revise the existing academic report using the feedback. Do not "
            "add citations, sources, statistics or external evidence."
        )
        required_headings = [
            section["heading"] for section in plan["sections"]
        ]
        user_instruction = f"""
        Research question: {question}
        Required headings: {required_headings}
        Feedback: {review.get("feedback", [])}
        Existing report: {initial_report}

        Return only a revised 350 to 450 word report with exactly four
        level-two headings and a complete conclusion.
        """
        revision_raw = generate_text(
            system_instruction, user_instruction, 650
        )
        candidate = clean_generated_report(revision_raw)
        candidate, revision_heading_changes = normalise_no_rag_heading_levels(
            candidate, required_headings
        )
        if no_rag_report_checks(candidate)["basic_structure_passed"]:
            final_report = candidate
            revision_accepted = True
    else:
        revision_heading_changes = []

    finished = datetime.now(timezone.utc)
    return {
        "condition": "B_agentic_no_RAG",
        "question": question,
        "plan": plan,
        "planner_raw_output": planner_raw,
        "initial_report_text": initial_report,
        "writer_raw_output": writer_raw,
        "initial_heading_level_normalisations": initial_heading_changes,
        "review": review,
        "revision_raw_output": revision_raw,
        "revision_accepted": revision_accepted,
        "revision_heading_level_normalisations": revision_heading_changes,
        "report_text": final_report,
        "checks": no_rag_report_checks(final_report),
        "started_utc": started.isoformat(),
        "finished_utc": finished.isoformat(),
        "duration_seconds": round(
            (finished - started).total_seconds(), 2
        ),
    }

In [ ]:
def section_claim_metrics(audit_rows, heading):
    """Calculate claim metrics for one section."""
    return calculate_claim_metrics(
        [row for row in audit_rows if row["section"] == heading]
    )


def claim_grounding_revision_agent(
    research_question,
    section_heading,
    original_section,
    selected_evidence,
    flagged_rows,
):
    """Make the single revision attempt allowed by the frozen workflow."""
    allowed_pmcids = sorted(
        {item["pmcid"] for item in selected_evidence}
    )
    evidence_text = format_candidate_evidence(selected_evidence)
    flagged_text = "\n".join(
        f'- {row["status"]}: {row["sentence"]}'
        for row in flagged_rows
    )
    system_instruction = (
        "Repair factual grounding using only the supplied PMC passages. "
        "Every factual sentence must be a direct paraphrase of its cited "
        "passage. Return only the revised Markdown section."
    )
    user_instruction = f"""
    Research question: {research_question}
    Required heading: ## {section_heading}
    Allowed PMC identifiers: {allowed_pmcids}
    Approved evidence: {evidence_text}
    Original section: {original_section}
    Flagged claims: {flagged_text}

    Rewrite in 120 to 180 words and exactly {len(selected_evidence)}
    substantive paragraphs. Use only approved evidence, cite every factual
    sentence, end every paragraph with an allowed citation and add no source list.
    """
    raw_output = generate_text(
        system_instruction, user_instruction, WRITER_MAX_TOKENS
    )
    candidate = normalise_section_output(section_heading, raw_output)
    candidate, citation_changes = normalise_generated_citations(
        candidate, selected_evidence
    )
    section_checks = calculate_section_checks(candidate, selected_evidence)
    candidate_audit = audit_report_claims(candidate, selected_evidence)
    return {
        "candidate_section": candidate,
        "raw_output": raw_output,
        "citation_normalisations": citation_changes,
        "section_checks": section_checks,
        "claim_audit": candidate_audit,
    }


def apply_frozen_claim_verification(stage_result, research_question):
    """Audit the RAG report and retain only qualifying improvements."""
    initial_audit = audit_report_claims(
        stage_result["report_text"], stage_result["selected_evidence"]
    )
    report_sections = split_report_sections(stage_result["report_text"])
    evidence_by_chunk = {
        item["chunk_id"]: item
        for item in stage_result["selected_evidence"]
    }
    revision_traces = []
    final_sections = []

    for section_trace in stage_result["section_traces"]:
        heading = section_trace["heading"]
        original_section = report_sections[heading]
        section_evidence = [
            evidence_by_chunk[chunk_id]
            for chunk_id in section_trace["selected_chunk_ids"]
            if chunk_id in evidence_by_chunk
        ]
        flagged_rows = [
            row
            for row in initial_audit["rows"]
            if row["section"] == heading and row["status"] != "supported"
        ]
        original_metrics = section_claim_metrics(
            initial_audit["rows"], heading
        )

        if not flagged_rows:
            final_sections.append(original_section)
            revision_traces.append(
                {
                    "heading": heading,
                    "revision_attempted": False,
                    "revision_accepted": False,
                    "reason": "No claim was flagged by the verifier.",
                    "original_claim_metrics": original_metrics,
                }
            )
            continue

        revision = claim_grounding_revision_agent(
            research_question,
            heading,
            original_section,
            section_evidence,
            flagged_rows,
        )
        candidate_metrics = revision["claim_audit"]["metrics"]
        improved = (
            candidate_metrics["claim_support_rate"]
            > original_metrics["claim_support_rate"]
        )
        accepted = bool(
            revision["section_checks"]["section_gate_passed"]
            and improved
            and candidate_metrics["claim_support_rate"]
            >= CLAIM_SUPPORT_GATE
            and candidate_metrics["invalid_citation_claims"] == 0
            and candidate_metrics["contradicted_claims"] == 0
        )
        final_sections.append(
            revision["candidate_section"] if accepted else original_section
        )
        revision_traces.append(
            {
                "heading": heading,
                "revision_attempted": True,
                "revision_accepted": accepted,
                "original_claim_metrics": original_metrics,
                "candidate_claim_metrics": candidate_metrics,
                "candidate_section_checks": revision["section_checks"],
                "flagged_claims": flagged_rows,
                "candidate_section": revision["candidate_section"],
                "raw_output": revision["raw_output"],
                "citation_normalisations": revision[
                    "citation_normalisations"
                ],
            }
        )

    final_report = (
        f'# {stage_result["plan"]["report_title"]}\n\n'
        + "\n\n".join(final_sections)
        + "\n\n## Evidence sources\n\n"
        + build_evidence_source_list(stage_result["selected_evidence"])
    )
    final_audit = audit_report_claims(
        final_report, stage_result["selected_evidence"]
    )
    final_checks = calculate_report_checks(
        final_report,
        {item["pmcid"] for item in stage_result["selected_evidence"]},
    )
    return {
        "initial_report_text": stage_result["report_text"],
        "initial_claim_audit": initial_audit,
        "revision_traces": revision_traces,
        "final_report_text": final_report,
        "final_claim_audit": final_audit,
        "final_report_checks": final_checks,
        "combined_gate_passed": bool(
            final_checks["structural_gate_passed"]
            and final_audit["metrics"]["claim_grounding_gate_passed"]
        ),
    }


def run_condition_c(question):
    """Run the frozen Agentic RAG workflow and claim verifier."""
    started = datetime.now(timezone.utc)
    stage_result = orchestrate_report(question)
    verification = apply_frozen_claim_verification(stage_result, question)
    finished = datetime.now(timezone.utc)
    return {
        "condition": "C_agentic_RAG_claim_verified",
        "question": question,
        "stage_4_trace": stage_result,
        "claim_verification": verification,
        "report_text": verification["final_report_text"],
        "started_utc": started.isoformat(),
        "finished_utc": finished.isoformat(),
        "duration_seconds": round(
            (finished - started).total_seconds(), 2
        ),
    }

## 10. Apply the frozen cross-theme functions

These later definitions replace earlier development versions of the planner and evidence reviewer. They are retained from the formal notebook so the live demo uses the same final cross-theme logic.

In [ ]:
# The development prototype screened evidence specifically for LLM terms.
# Before any formal outputs were generated, this was generalised so all five
# frozen healthcare themes are treated in the same way.
FORMAL_FREEZE_ID = "STAGE_5_FORMAL_GENERATION_V1_2026-09-02"


def planner_agent(research_question):
    """Create a four-section plan that works for any healthcare-AI question."""
    system_instruction = (
        "You are the planner in an academic healthcare-AI research workflow. "
        "Plan a concise evidence-based report, not clinical advice. Return JSON only."
    )
    user_instruction = f"""
    Research question: {research_question}

    Return this structure:
    {{
      "report_title": "short title",
      "sections": [
        {{"heading": "section heading", "retrieval_query": "focused evidence query"}}
      ]
    }}

    Requirements:
    - exactly {MAX_REPORT_SECTIONS} sections;
    - cover context/applications, potential benefits or evidence, risks/limitations,
      and safeguards/conclusion;
    - each retrieval query must be specific to the research question and understandable alone;
    - do not include references because retrieval happens next.
    """
    raw_output = generate_text(
        system_instruction, user_instruction, PLANNER_MAX_TOKENS
    )
    plan = extract_json_object(raw_output)
    valid = (
        isinstance(plan, dict)
        and isinstance(plan.get("report_title"), str)
        and isinstance(plan.get("sections"), list)
        and len(plan["sections"]) == MAX_REPORT_SECTIONS
        and all(
            isinstance(section, dict)
            and section.get("heading")
            and section.get("retrieval_query")
            for section in plan["sections"]
        )
    )
    if not valid:
        plan = {
            "report_title": "Evidence-based analysis of the healthcare AI question",
            "sections": [
                {
                    "heading": "Context and applications",
                    "retrieval_query": research_question + " context and applications",
                },
                {
                    "heading": "Potential benefits and evidence",
                    "retrieval_query": research_question + " benefits supporting evidence",
                },
                {
                    "heading": "Risks and limitations",
                    "retrieval_query": research_question + " risks limitations barriers",
                },
                {
                    "heading": "Safeguards and conclusion",
                    "retrieval_query": research_question + " safeguards monitoring conclusion",
                },
            ],
            "parse_fallback_used": True,
        }
    else:
        plan["parse_fallback_used"] = False
    return plan, raw_output


def evidence_reviewer_agent(section_heading, retrieval_query, candidates):
    """Select relevant evidence without restricting the healthcare-AI theme."""
    system_instruction = (
        "You review PMC evidence for an academic healthcare-AI report. Select "
        "only passages that directly support the section and evidence question. "
        "Return valid JSON only."
    )
    user_instruction = f"""
    Section: {section_heading}
    Evidence question: {retrieval_query}

    Candidate passages:
    {format_candidate_evidence(candidates)}

    Return:
    {{
      "selected_chunk_ids": ["chunk id", "chunk id"],
      "reason": "brief explanation",
      "limitations": "what the selected evidence does not establish"
    }}

    Select between 2 and 3 passages. Do not invent chunk IDs.
    """
    raw_output = generate_text(
        system_instruction, user_instruction, REVIEWER_MAX_TOKENS
    )
    decision = extract_json_object(raw_output)
    valid_ids = {item["chunk_id"] for item in candidates}
    if isinstance(decision, dict):
        selected_ids = [
            chunk_id
            for chunk_id in decision.get("selected_chunk_ids", [])
            if chunk_id in valid_ids
        ][:3]
    else:
        selected_ids = []

    fallback_used = len(selected_ids) < 2
    if fallback_used:
        selected_ids = [item["chunk_id"] for item in candidates[:3]]
    selected = [
        item for item in candidates if item["chunk_id"] in selected_ids
    ]
    return {
        "selected_evidence": selected,
        "raw_output": raw_output,
        "fallback_used": fallback_used,
        "screening_fallback_used": False,
        "screened_candidate_ids": [item["chunk_id"] for item in candidates],
        "screened_out_chunk_ids": [],
        "reason": decision.get("reason", "") if isinstance(decision, dict) else "",
        "limitations": (
            decision.get("limitations", "") if isinstance(decision, dict) else ""
        ),
    }


def no_rag_planner_agent(question):
    """Create a generic four-section plan for formal Condition B."""
    system_instruction = (
        "You are the planner in an academic healthcare-AI report workflow. "
        "Plan a balanced report using internal model knowledge only. Return JSON only."
    )
    user_instruction = f"""
    Research question: {question}

    Return exactly:
    {{
      "report_title": "short title",
      "sections": [
        {{"heading": "section heading", "purpose": "section purpose"}}
      ]
    }}

    Include exactly four sections covering context/applications, potential
    benefits or evidence, risks/limitations, and safeguards/conclusion.
    Do not include citations, references or retrieval instructions.
    """
    raw_output = generate_text(
        system_instruction, user_instruction, PLANNER_MAX_TOKENS
    )
    plan = extract_json_object(raw_output)
    valid = (
        isinstance(plan, dict)
        and isinstance(plan.get("report_title"), str)
        and isinstance(plan.get("sections"), list)
        and len(plan["sections"]) == MAX_REPORT_SECTIONS
        and all(
            isinstance(section, dict)
            and section.get("heading")
            and section.get("purpose")
            for section in plan["sections"]
        )
    )
    if not valid:
        plan = {
            "report_title": "Analysis of the healthcare AI question",
            "sections": [
                {
                    "heading": "Context and applications",
                    "purpose": "Explain the relevant context and uses.",
                },
                {
                    "heading": "Potential benefits and evidence",
                    "purpose": "Discuss potential benefits and supporting considerations.",
                },
                {
                    "heading": "Risks and limitations",
                    "purpose": "Discuss technical, clinical and ethical limitations.",
                },
                {
                    "heading": "Safeguards and conclusion",
                    "purpose": "Present safeguards and a balanced conclusion.",
                },
            ],
            "parse_fallback_used": True,
        }
    else:
        plan["parse_fallback_used"] = False
    return plan, raw_output

print("Formal cross-theme planner and evidence reviewer loaded")
print("Formal freeze ID:", FORMAL_FREEZE_ID)

## 11. Preview retrieval for my question

This preview is a real retrieval call. The full workflow below searches again using the planner's section queries, so its final selected passages can differ from this preview.


In [ ]:
import pandas as pd
from IPython.display import display, Markdown

preview_evidence = hybrid_retrieve(RESEARCH_QUESTION)
display(pd.DataFrame(preview_evidence)[
    ["rank", "pmcid", "title", "hybrid_rrf_score", "source_url", "text"]
])


## 12. Run one fresh report

This cell calls the model; it does not read a saved report. Each run has a new directory. The formal 45 reports are not overwritten or added to. If a run fails, the error is recorded and re-raised; there is no silent saved-answer fallback.


In [ ]:
def run_live_demo(question, include_baselines=False):
    """Run one new question and save each completed condition immediately."""
    import uuid
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
    run_dir = OUTPUT_DIR / run_id
    run_dir.mkdir(parents=True, exist_ok=False)
    metadata = {
        "run_id": run_id, "run_kind": "fresh_live_demo_not_formal_evaluation",
        "question": question, "seed": RANDOM_SEED,
        "generator": GENERATOR_MODEL_NAME, "generator_revision": GENERATOR_MODEL_REVISION,
        "nli_model": NLI_MODEL_NAME, "nli_revision": NLI_MODEL_REVISION,
        "input_sha256": source_archive_sha256, "formal_implementation": FORMAL_FREEZE_ID,
        "packages": {name: importlib.metadata.version(name) for name in
                     ["torch", "transformers", "accelerate", "sentence-transformers", "bitsandbytes"]},
        "completed_conditions": [], "status": "running"
    }
    def save_metadata():
        (run_dir / "run_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    save_metadata()
    results = {}
    jobs = [("A", run_condition_a), ("B", run_condition_b)] if include_baselines else []
    jobs.append(("C", run_condition_c))
    try:
        for label, runner in jobs:
            random.seed(RANDOM_SEED)
            np.random.seed(RANDOM_SEED)
            torch.manual_seed(RANDOM_SEED)
            torch.cuda.manual_seed_all(RANDOM_SEED)
            print("Running condition", label, "for a NEW demo answer...", flush=True)
            result = runner(question)
            results[label] = result
            (run_dir / f"condition_{label}_trace.json").write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
            (run_dir / f"condition_{label}_report.md").write_text(result["report_text"], encoding="utf-8")
            metadata["completed_conditions"].append(label)
            save_metadata()
            gc.collect()
            torch.cuda.empty_cache()
    except Exception as error:
        metadata.update(status="failed", error=repr(error))
        save_metadata()
        print("Demo stopped. Completed outputs, if any, are in:", run_dir)
        raise
    metadata["status"] = "completed"
    save_metadata()
    return results, run_dir

demo_results, demo_dir = run_live_demo(RESEARCH_QUESTION, RUN_BASELINES)
print("Fresh demo saved:", demo_dir)


## 13. Show the new report, evidence and checks

For the professor: explain one cited claim, open its source, and compare it with the retrieved passage. Check both successful and failed claims; do not only show favourable output.

In [ ]:
current = demo_results["C"]
display(Markdown("# Fresh live demo — Condition C"))
display(Markdown(current["report_text"]))
print("Generation and verification seconds:", current["duration_seconds"])

trace = current["stage_4_trace"]
verification = current["claim_verification"]
print("Planner output:")
display(trace["plan"])
print("Selected source passages:")
display(pd.DataFrame(trace["selected_evidence"])[["chunk_id", "pmcid", "title", "text", "source_url"]])
print("Structural checks:")
display(verification["final_report_checks"])
print("Automated claim metrics (not human evaluation):")
display(verification["final_claim_audit"]["metrics"])
display(pd.DataFrame(verification["final_claim_audit"]["rows"]))
print("Conditional claim-revision decisions:")
display(pd.DataFrame([{k: v for k, v in row.items() if k in
    ["heading", "revision_attempted", "revision_accepted", "reason"]}
    for row in verification["revision_traces"]]))
print("Combined automated gate passed:", verification["combined_gate_passed"])
print("Full prompts/outputs and section traces are in the downloaded trace JSON.")
for label in ["A", "B"]:
    if label in demo_results:
        display(Markdown(f"# Optional fresh baseline {label}"))
        display(Markdown(demo_results[label]["report_text"]))


## 14. Download this demo run

Keep this ZIP as a live-demo record. It is not a replacement for frozen formal evaluation evidence.

In [ ]:
demo_checksums = {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                  for p in demo_dir.iterdir() if p.is_file() and p.name != "checksums_sha256.json"}
(demo_dir / "checksums_sha256.json").write_text(json.dumps(demo_checksums, indent=2), encoding="utf-8")
demo_archive = shutil.make_archive(str(demo_dir) + "_evidence", "zip", demo_dir)
print("New demo archive:", demo_archive)
files.download(demo_archive)


## What I explain, and what I do not claim

- The stage notebooks are the complete research pipeline; this notebook is the one-question entry point for its generation stage.
- The underlying models are pretrained. I have not trained a new foundation model.
- Python functions coordinate model roles; this is not a LangChain implementation.
- Passing a citation/claim check does not prove factual accuracy. Human review is still valuable and has not been replaced by these models.
- A live answer can differ from a frozen answer across hardware/software runs. Do not mix this demo into the original comparison.
- I used AI tools to support brainstorming, code organisation, debugging and clearer explanations. I checked the code and I remain responsible for understanding and explaining the work. I will describe this support in my assessment declaration in line with the university guidance.

After the live demonstration, open `Adarsh_Konderu_Final_Results_And_Graphs.ipynb` to discuss the actual frozen experiment and its limitations.
